# Schema-Miner with a Local Hugging Face Model on GPU

This tutorial demonstrates how to use **Schema-Miner** with a Hugging Face model running locally on a CUDA-enabled GPU.

We use:

- **Schema-Miner** for scientific schema mining and iterative schema refinement
- **Hugging Face** for local model inference
- **mistralai/Ministral-3-3B-Instruct-2512** as the example large language model
- **GPU acceleration** for local inference

This notebook demonstrates the complete three-stage Schema-Miner workflow:

1. **Stage 1 — Initial Schema Mining**
2. **Stage 2 — Preliminary Schema Refinement**
3. **Stage 3 — Final Schema Refinement**

> This tutorial is intended for environments with a CUDA-enabled GPU. A separate tutorial demonstrates Schema-Miner with an API-based model provider, which does not require a local GPU.

## 1. Installation

Install Schema-Miner directly from PyPI:

```bash
pip install schema-miner
```

In [11]:
%pip install -U schema-miner

Note: you may need to restart the kernel to use updated packages.


## 2. Verify the Environment

Before running Schema-Miner, verify that the notebook is connected to the expected GPU-enabled environment.

In [12]:
import os
import socket

print("Host:", socket.gethostname())
print("SLURM_JOB_ID:", os.environ.get("SLURM_JOB_ID"))
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

Host: gpu-l40s-02
SLURM_JOB_ID: 29942
CUDA_VISIBLE_DEVICES: 0


In [13]:
import importlib.metadata
import torch

print("Schema-Miner version:", importlib.metadata.version("schema-miner"))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB"
    )

Schema-Miner version: 3.2.5
CUDA available: True
GPU: NVIDIA L40S
GPU memory: 44.4 GB


## 3. Stage 1 — Initial Schema Mining

Stage 1 generates an initial schema from a process specification document using a local Hugging Face model.

This tutorial assumes the following structure:

```text
data/
└── stage1/
    ├── process.txt
    ├── process-description.pdf
    ├── feedback/
    └── schema/
```

The `process.txt` file contains the process name and process description:

```text
process name: <process name>
process description: <process description>
```

The process specification is provided as a single PDF in `data/stage1/`.

The `schema/` directory stores the schema generated by Stage 1. The `feedback/` directory is used later to provide expert feedback for Stage 2.

The example model used in this tutorial is `mistralai/Ministral-3-3B-Instruct-2512`. You may enter another Hugging Face model when prompted.

In [17]:
import os
import subprocess
from getpass import getpass
from pathlib import Path

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

STAGE1_DIR = Path("data/stage1")

DEFAULT_MODEL = "mistralai/Ministral-3-8B-Instruct-2512"

model = input(
    f"Hugging Face model [{DEFAULT_MODEL}]: "
).strip() or DEFAULT_MODEL

# Optional authentication, e.g. for gated Hugging Face models
hf_token = getpass(
    "Hugging Face access token (press Enter if not required): "
)

# ------------------------------------------------------------------
# Read process information
# ------------------------------------------------------------------

process_txt = STAGE1_DIR / "process.txt"

if not process_txt.exists():
    raise FileNotFoundError(f"Missing process file: {process_txt}")

text = process_txt.read_text(encoding="utf-8").strip()
lines = text.splitlines()

process_name = None
description_lines = []
in_description = False

for line in lines:
    stripped = line.strip()
    lower = stripped.lower()

    if lower.startswith("process name:"):
        process_name = stripped.split(":", 1)[1].strip()
        in_description = False

    elif lower.startswith("process description:"):
        description_lines.append(
            stripped.split(":", 1)[1].strip()
        )
        in_description = True

    elif in_description:
        description_lines.append(stripped)

process_description = "\n".join(description_lines).strip()

if not process_name:
    raise ValueError("Missing 'process name:' in process.txt")

if not process_description:
    raise ValueError("Missing 'process description:' in process.txt")

# ------------------------------------------------------------------
# Locate the process specification
# ------------------------------------------------------------------

pdfs = sorted(STAGE1_DIR.glob("*.pdf"))

if len(pdfs) != 1:
    raise ValueError(
        f"Expected exactly one PDF in {STAGE1_DIR}, found {len(pdfs)}."
    )

specification_pdf = pdfs[0]

# ------------------------------------------------------------------
# Output directory
# ------------------------------------------------------------------

results_dir = STAGE1_DIR / "schema"
results_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Runtime environment for Schema-Miner
# ------------------------------------------------------------------

env = os.environ.copy()

env.update({
    "LLM_PROVIDER": "HUGGINGFACE",
    "LLM_MODEL": model,
    "HUGGINGFACE_USE_LOCAL": "True",
    "PROCESS_NAME": process_name,
    "PROCESS_DESCRIPTION": process_description,
    "STAGE1_SPECS_PATH": str(specification_pdf),
    "STAGE2_PAPERS_PATH": "",
    "STAGE3_PAPERS_PATH": "",
    "RESULTS_PATH": str(results_dir),
})

if hf_token:
    env["HuggingFace_Access_Token"] = hf_token

del hf_token

# ------------------------------------------------------------------
# Run Stage 1
# ------------------------------------------------------------------

print(f"Process:       {process_name}")
print(f"Model:         {model}")
print(f"Specification: {specification_pdf}")
print(f"Results:       {results_dir}")

subprocess.run(
    ["schema-miner", "--stage", "1"],
    env=env,
    check=True,
)

Process:       Metal-organic cages synthesis
Model:         mistralai/Ministral-3-8B-Instruct-2512
Specification: data/stage1/process-description.pdf
Results:       data/stage1/schema
Running SCHEMA-MINER -- Stage 1: Initial Schema Extraction


2026-08-11 11:22:48,587 - LLMs4SchemaDiscovery Framework -- A Human-in-the-Loop Workflow for Scientific Schema Mining with Large Language Models for Metal-organic cages synthesis process
2026-08-11 11:22:48,587 - Stage 1: Initial Schema Mining
2026-08-11 11:22:48,587 - Reading the process specification document...
2026-08-11 11:22:48,587 - Extracting text from the PDF: process-description.pdf
2026-08-11 11:22:48,878 - PDF parsed successfully
2026-08-11 11:22:48,878 - Performing LLM (mistralai/Ministral-3-8B-Instruct-2512) Inference to extract schema...
2026-08-11 11:22:48,878 - Using local HuggingFace model: mistralai/Ministral-3-8B-Instruct-2512 for LLM inference
Loading weights: 100%|██████████| 1007/1007 [00:01<00:00, 816.92it/s]
[transformers] The model 'Mistral3ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForC

Stage 1 completed. Initial schema saved to data/stage1/schema


CompletedProcess(args=['schema-miner', '--stage', '1'], returncode=0)

## 4. Stage 2 — Preliminary Schema Refinement

Stage 2 refines the initial schema using scientific papers together with expert feedback from the preceding schema-mining run.

The papers may be organized into **one or more batches**. A batch can contain a single paper or several papers, depending on how frequently expert review should be incorporated.

For example, this tutorial uses two batches:

```text
data/
└── stage2/
    ├── batch1/
    │   ├── paper-1.pdf
    │   ├── paper-2.pdf
    │   └── ...
    ├── batch2/
    │   ├── paper-6.pdf
    │   ├── paper-7.pdf
    │   └── ...
    ├── feedback-batch1/
    ├── feedback-batch2/
    ├── schema-batch1/
    └── schema-batch2/
```

The refinement proceeds iteratively:

```text
Stage 1 schema
      +
Stage 1 expert feedback
      +
Stage 2 / batch1 papers
      ↓
Stage 2 / schema-batch1
      ↓
expert review
      ↓
Stage 2 / feedback-batch1
      +
Stage 2 / batch2 papers
      ↓
Stage 2 / schema-batch2
```

After each batch, inspect the resulting schema and place the corresponding expert feedback in the appropriate `feedback-batchN/` directory before running the next batch.

The number of batches is not fixed. For example, the papers could instead be organized into three batches, or even processed one paper at a time. The important requirement is that **each new batch uses the schema and expert feedback resulting from review of the previous run**.

Run the following cell once for each Stage 2 batch. The available batches are detected automatically from the `data/stage2/` directory.

In [20]:
# Stage 2 — Preliminary Schema Refinement

import os
import subprocess
from pathlib import Path

STAGE1_DIR = Path("data/stage1")
STAGE2_DIR = Path("data/stage2")

# ------------------------------------------------------------------
# Detect available batches
# ------------------------------------------------------------------

available_batches = sorted(
    int(path.name.replace("batch", ""))
    for path in STAGE2_DIR.glob("batch*")
    if path.is_dir() and path.name.replace("batch", "").isdigit()
)

if not available_batches:
    raise FileNotFoundError(
        f"No batch directories found in {STAGE2_DIR}"
    )

print("Available Stage 2 batches:", available_batches)

batch = int(
    input(
        f"Stage 2 batch to run {available_batches}: "
    ).strip()
)

if batch not in available_batches:
    raise ValueError(
        f"Batch {batch} not found. Available batches: {available_batches}"
    )

# ------------------------------------------------------------------
# Resolve model-specific filenames
# ------------------------------------------------------------------

model_file = model.replace("/", "-")

# ------------------------------------------------------------------
# Resolve schema and expert feedback from the preceding run
# ------------------------------------------------------------------

if batch == 1:
    # First Stage 2 batch starts from the Stage 1 result
    schema_path = (
        STAGE1_DIR
        / "schema"
        / f"{model_file}.json"
    )

    feedback_path = (
        STAGE1_DIR
        / "feedback"
        / f"{model_file}.txt"
    )

else:
    # Later batches continue from the preceding Stage 2 batch
    previous_batch = batch - 1

    schema_path = (
        STAGE2_DIR
        / f"schema-batch{previous_batch}"
        / f"{model_file}.json"
    )

    feedback_path = (
        STAGE2_DIR
        / f"feedback-batch{previous_batch}"
        / f"{model_file}.txt"
    )

# ------------------------------------------------------------------
# Current papers and output directory
# ------------------------------------------------------------------

papers_dir = STAGE2_DIR / f"batch{batch}"

results_dir = STAGE2_DIR / f"schema-batch{batch}"
results_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Validate inputs
# ------------------------------------------------------------------

if not schema_path.exists():
    raise FileNotFoundError(
        f"Schema from the preceding run not found: {schema_path}"
    )

if not feedback_path.exists():
    raise FileNotFoundError(
        f"Expert feedback from the preceding run not found: {feedback_path}"
    )

pdfs = sorted(papers_dir.glob("*.pdf"))

if not pdfs:
    raise FileNotFoundError(
        f"No PDF papers found in {papers_dir}"
    )

# ------------------------------------------------------------------
# Runtime environment for Schema-Miner
# ------------------------------------------------------------------

# Preserve the environment established during Stage 1, including
# Hugging Face authentication if it was supplied there.
env = dict(env)

env.update({
    "LLM_PROVIDER": "HUGGINGFACE",
    "LLM_MODEL": model,
    "HUGGINGFACE_USE_LOCAL": "True",
    "PROCESS_NAME": process_name,
    "PROCESS_DESCRIPTION": process_description,
    "STAGE1_SPECS_PATH": "",
    "STAGE2_PAPERS_PATH": str(papers_dir),
    "STAGE3_PAPERS_PATH": "",
    "RESULTS_PATH": str(results_dir),
})

# ------------------------------------------------------------------
# Run Stage 2
# ------------------------------------------------------------------

command = [
    "schema-miner",
    "--stage", "2",
    "--schema", str(schema_path),
    "--expert-feedback", str(feedback_path),
    "--papers", "all",
]

log_path = results_dir / f"stage2-batch{batch}.log"

print(f"Running Stage 2 — Batch {batch}")
print(f"Model:           {model}")
print(f"Input schema:    {schema_path}")
print(f"Expert feedback: {feedback_path}")
print(f"Papers:          {papers_dir} ({len(pdfs)} PDFs)")
print(f"Results:         {results_dir}")
print()

try:
    process = subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()

    # Save the complete CLI output for reproducibility/debugging
    log_path.write_text(
        "".join(output_lines),
        encoding="utf-8",
    )

    if return_code == 0:
        print("\n✓ Stage 2 completed successfully.")
        print(f"Results: {results_dir}")

    else:
        print("\n" + "=" * 70)
        print(f"✗ Stage 2 — Batch {batch} did not complete successfully.")
        print(f"Schema-Miner exited with status code {return_code}.")
        print(f"Full log: {log_path}")
        print(
            "Review the Schema-Miner output above for the underlying error, "
            "correct the issue, and rerun this cell."
        )
        print("=" * 70)

except FileNotFoundError:
    print(
        "\n✗ The `schema-miner` command could not be found. "
        "Make sure Schema-Miner is installed in the current notebook environment."
    )

except KeyboardInterrupt:
    if "process" in locals() and process.poll() is None:
        process.terminate()
        process.wait()

    print("\n⚠ Stage 2 was interrupted by the user.")

Available Stage 2 batches: [1, 2]


Running Stage 2 — Batch 2
Model:           mistralai/Ministral-3-8B-Instruct-2512
Input schema:    data/stage2/schema-batch1/mistralai-Ministral-3-8B-Instruct-2512.json
Expert feedback: data/stage2/feedback-batch1/mistralai-Ministral-3-8B-Instruct-2512.txt
Papers:          data/stage2/batch2 (2 PDFs)
Results:         data/stage2/schema-batch2

Running SCHEMA-MINER -- Stage 2: Preliminary Schema Refinement
Total papers: 2 | Batch size: 2 | Total batches: 1

Batch 1/1:
Processing paper 1/2: data/stage2/batch2/Zhu_2024.pdf
2026-08-11 11:55:11,735 - LLMs4SchemaDiscovery Framework -- A Human-in-the-Loop Workflow for Scientific Schema Mining with Large Language Models for Metal-organic cages synthesis process
2026-08-11 11:55:11,736 - Stage 2: Preliminary Schema Refinement
2026-08-11 11:55:11,736 - Reading the schema...
2026-08-11 11:55:11,738 - Reading the domain expert review on the schema...
2026-08-11 11:55:11,740 - Reading the scientific paper...
2026-08-11 11:55:11,740 - Extracting tex

## 5. Stage 3 — Final Schema Refinement

Stage 3 performs the final refinement of the schema using a broader set of scientific papers together with expert feedback from the preceding schema-mining run.

As in Stage 2, the papers may be organized into **one or more batches**. A batch may contain a single paper or several papers, depending on how frequently expert review should be incorporated.

For example, this tutorial uses two batches:

```text
data/
└── stage3/
    ├── batch1/
    │   ├── paper-1.pdf
    │   ├── paper-2.pdf
    │   └── ...
    ├── batch2/
    │   ├── paper-6.pdf
    │   ├── paper-7.pdf
    │   └── ...
    ├── feedback-batch1/
    ├── schema-batch1/
    └── schema-batch2/
```

Stage 3 begins from the **final Stage 2 schema and its corresponding expert feedback**:

```text
Final Stage 2 schema
      +
Final Stage 2 expert feedback
      +
Stage 3 / batch1 papers
      ↓
Stage 3 / schema-batch1
      ↓
expert review
      ↓
Stage 3 / feedback-batch1
      +
Stage 3 / batch2 papers
      ↓
Stage 3 / schema-batch2
```

For subsequent Stage 3 batches, each run uses the schema and expert feedback resulting from review of the preceding Stage 3 batch.

The number of batches is not fixed. The papers may be divided into two batches, three batches, or processed one paper at a time. The important requirement is that **each new batch uses the schema and expert feedback resulting from review of the previous run**.

Expert feedback only needs to be created when another refinement batch will follow. For example, if `batch2` is the final Stage 3 batch, no `feedback-batch2/` directory is required.

Run the following cell once for each Stage 3 batch. The available batches are detected automatically from the `data/stage3/` directory.

In [23]:
# Stage 3 — Final Schema Refinement

import subprocess
from pathlib import Path

STAGE2_DIR = Path("data/stage2")
STAGE3_DIR = Path("data/stage3")

# ------------------------------------------------------------------
# Detect available Stage 3 batches
# ------------------------------------------------------------------

available_batches = sorted(
    int(path.name.replace("batch", ""))
    for path in STAGE3_DIR.glob("batch*")
    if path.is_dir() and path.name.replace("batch", "").isdigit()
)

if not available_batches:
    raise FileNotFoundError(
        f"No batch directories found in {STAGE3_DIR}"
    )

print("Available Stage 3 batches:", available_batches)

batch = int(
    input(
        f"Stage 3 batch to run {available_batches}: "
    ).strip()
)

if batch not in available_batches:
    raise ValueError(
        f"Batch {batch} not found. Available batches: {available_batches}"
    )

# ------------------------------------------------------------------
# Resolve model-specific filenames
# ------------------------------------------------------------------

model_file = model.replace("/", "-")

# ------------------------------------------------------------------
# Resolve schema and expert feedback from the preceding run
# ------------------------------------------------------------------

if batch == 1:
    # Stage 3 starts from the latest completed Stage 2 batch.
    stage2_checkpoints = []

    for schema_dir in STAGE2_DIR.glob("schema-batch*"):
        suffix = schema_dir.name.replace("schema-batch", "")

        if not suffix.isdigit():
            continue

        stage2_batch = int(suffix)

        schema_candidate = (
            schema_dir / f"{model_file}.json"
        )

        feedback_candidate = (
            STAGE2_DIR
            / f"feedback-batch{stage2_batch}"
            / f"{model_file}.txt"
        )

        if schema_candidate.exists() and feedback_candidate.exists():
            stage2_checkpoints.append(
                (
                    stage2_batch,
                    schema_candidate,
                    feedback_candidate,
                )
            )

    if not stage2_checkpoints:
        raise FileNotFoundError(
            "No completed Stage 2 checkpoint with both schema "
            "and expert feedback was found."
        )

    (
        final_stage2_batch,
        schema_path,
        feedback_path,
    ) = max(stage2_checkpoints, key=lambda item: item[0])

    print(
        f"Starting Stage 3 from Stage 2 Batch {final_stage2_batch}"
    )

else:
    # Later Stage 3 batches continue from the preceding Stage 3 batch.
    previous_batch = batch - 1

    schema_path = (
        STAGE3_DIR
        / f"schema-batch{previous_batch}"
        / f"{model_file}.json"
    )

    feedback_path = (
        STAGE3_DIR
        / f"feedback-batch{previous_batch}"
        / f"{model_file}.txt"
    )

# ------------------------------------------------------------------
# Current papers and output directory
# ------------------------------------------------------------------

papers_dir = STAGE3_DIR / f"batch{batch}"

results_dir = STAGE3_DIR / f"schema-batch{batch}"
results_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Validate inputs
# ------------------------------------------------------------------

if not schema_path.exists():
    raise FileNotFoundError(
        f"Schema from the preceding run not found: {schema_path}"
    )

if not feedback_path.exists():
    raise FileNotFoundError(
        f"Expert feedback from the preceding run not found: {feedback_path}"
    )

pdfs = sorted(papers_dir.glob("*.pdf"))

if not pdfs:
    raise FileNotFoundError(
        f"No PDF papers found in {papers_dir}"
    )

# ------------------------------------------------------------------
# Runtime environment for Schema-Miner
# ------------------------------------------------------------------

# Preserve the environment established during the previous stages,
# including Hugging Face authentication if it was supplied in Stage 1.
env = dict(env)

env.update({
    "LLM_PROVIDER": "HUGGINGFACE",
    "LLM_MODEL": model,
    "HUGGINGFACE_USE_LOCAL": "True",
    "PROCESS_NAME": process_name,
    "PROCESS_DESCRIPTION": process_description,
    "STAGE1_SPECS_PATH": "",
    "STAGE2_PAPERS_PATH": "",
    "STAGE3_PAPERS_PATH": str(papers_dir),
    "RESULTS_PATH": str(results_dir),
})

# ------------------------------------------------------------------
# Run Stage 3
# ------------------------------------------------------------------

print(f"Running Stage 3 — Batch {batch}")
print(f"Model:           {model}")
print(f"Input schema:    {schema_path}")
print(f"Expert feedback: {feedback_path}")
print(f"Papers:          {papers_dir} ({len(pdfs)} PDFs)")
print(f"Results:         {results_dir}")

subprocess.run(
    [
        "schema-miner",
        "--stage", "3",
        "--schema", str(schema_path),
        "--expert-feedback", str(feedback_path),
        "--papers", "all",
    ],
    env=env,
    check=True,
)

Available Stage 3 batches: [1, 2]


Running Stage 3 — Batch 2
Model:           mistralai/Ministral-3-8B-Instruct-2512
Input schema:    data/stage3/schema-batch1/mistralai-Ministral-3-8B-Instruct-2512.json
Expert feedback: data/stage3/feedback-batch1/mistralai-Ministral-3-8B-Instruct-2512.txt
Papers:          data/stage3/batch2 (3 PDFs)
Results:         data/stage3/schema-batch2
Running SCHEMA-MINER -- Stage 3: Finalize Schema Refinement
Total papers: 3 | Batch size: 3 | Total batches: 1

Batch 1/1:
Processing paper 1/3: data/stage3/batch2/chemrxiv-2021-tlvxw.pdf


2026-08-11 12:53:03,288 - LLMs4SchemaDiscovery Framework -- A Human-in-the-Loop Workflow for Scientific Schema Mining with Large Language Models for Metal-organic cages synthesis process
2026-08-11 12:53:03,288 - Stage 3: Finalize Schema Refinement
2026-08-11 12:53:03,288 - Reading the schema...
2026-08-11 12:53:03,291 - Reading the domain expert review on the schema...
2026-08-11 12:53:03,293 - Reading the scientific paper...
2026-08-11 12:53:03,293 - Extracting text from the PDF: chemrxiv-2021-tlvxw.pdf
2026-08-11 12:53:04,186 - PDF parsed successfully
2026-08-11 12:53:04,186 - Calling the completion API of the model...
2026-08-11 12:53:04,186 - Using local HuggingFace model: mistralai/Ministral-3-8B-Instruct-2512 for LLM inference
Loading weights: 100%|██████████| 1007/1007 [00:01<00:00, 817.21it/s]
[transformers] The model 'Mistral3ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 

Processing paper 2/3: data/stage3/batch2/chemrxiv.13625783.v1.pdf


2026-08-11 13:03:02,318 - PDF parsed successfully
2026-08-11 13:03:02,318 - Calling the completion API of the model...
2026-08-11 13:03:02,318 - Using local HuggingFace model: mistralai/Ministral-3-8B-Instruct-2512 for LLM inference
Loading weights: 100%|██████████| 1007/1007 [00:01<00:00, 816.77it/s]
[transformers] The model 'Mistral3ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmFor

Processing paper 3/3: data/stage3/batch2/Chemistry A European J - 2024 - Ward - Chameleonic Cages  Encapsulation of Anionic  Neutral  and Cationic Guest Species.pdf


2026-08-11 13:12:57,343 - PDF parsed successfully
2026-08-11 13:12:57,343 - Calling the completion API of the model...
2026-08-11 13:12:57,343 - Using local HuggingFace model: mistralai/Ministral-3-8B-Instruct-2512 for LLM inference
Loading weights: 100%|██████████| 1007/1007 [00:01<00:00, 820.01it/s]
[transformers] The model 'Mistral3ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmFor

Completed batch 1/1

Stage 3 completed. Refined schemas saved to data/stage3/schema-batch2


CompletedProcess(args=['schema-miner', '--stage', '3', '--schema', 'data/stage3/schema-batch1/mistralai-Ministral-3-8B-Instruct-2512.json', '--expert-feedback', 'data/stage3/feedback-batch1/mistralai-Ministral-3-8B-Instruct-2512.txt', '--papers', 'all'], returncode=0)